# 双方向 LSTM と勾配クリッピング

**出典**　機械学習 4 分野の教科書 — 第 7 章 2-B CNN と RNN ／ 05 LSTM と GRU：ゲートで情報の流れを制御する

**このノートのパス**　`ml-code/07_cnn_rnn/02_sentiment_lstm.ipynb`

**このノートで確かめること**
教科書の断片（「前後の定義は省略」）を動く形に補う。
**可変長系列の梱包（pack_padded_sequence）**と、**勾配クリッピングを逆伝播と更新のあいだに挟む**位置を確かめる。

| 項目 | 内容 |
|---|---|
| 必要なパッケージ | `torch` |
| 実行時間の目安 | 約 30 秒 |
| 前提 | **このノート単体で完結する**（他のノートを先に実行する必要はない） |

In [2]:
# 未導入なら実行する（Colab など）
# !pip install -q torch

In [3]:
import torch, torch.nn as nn

class SentimentLSTM(nn.Module):
    """レビュー文の感情を 2 値分類する（教師あり × RNN）"""
    def __init__(self, vocab_size, emb_dim=128, hidden=256, num_classes=2):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            emb_dim, hidden,
            num_layers=2,
            batch_first=True,
            bidirectional=True,    # 前からと後ろからの両方向に読む
            dropout=0.2,
        )
        self.fc = nn.Linear(hidden * 2, num_classes)   # 双方向なので 2 倍

    def forward(self, x, lengths):
        e = self.emb(x)
        # 可変長系列を効率よく扱うための梱包。パディング部分を計算しない
        packed = nn.utils.rnn.pack_padded_sequence(
            e, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (h, _) = self.lstm(packed)
        # 最終層の順方向・逆方向の隠れ状態を結合
        h_last = torch.cat([h[-2], h[-1]], dim=1)
        return self.fc(h_last)

## 動かすためのダミーデータ — 可変長の系列をパディングして詰める

In [4]:
torch.manual_seed(0)
VOCAB, BATCH, MAXLEN = 5000, 16, 40

lengths = torch.randint(5, MAXLEN + 1, (BATCH,))          # 文ごとに長さが違う
x = torch.zeros(BATCH, MAXLEN, dtype=torch.long)          # 0 = パディング（padding_idx）
for i, L in enumerate(lengths):
    x[i, :L] = torch.randint(1, VOCAB, (int(L),))         # 1 以上が実トークン
y = torch.randint(0, 2, (BATCH,))

print("入力:", tuple(x.shape), "長さの例:", lengths[:6].tolist())

model = SentimentLSTM(vocab_size=VOCAB)
out = model(x, lengths)
print("出力:", tuple(out.shape), "← (バッチ, クラス数)")

入力: (16, 40) 長さの例: [13, 32, 34, 29, 12, 20]
出力: (16, 2) ← (バッチ, クラス数)


### 梱包が効いているか — パディング部分を変えても答えが変わらないことを確かめる

In [5]:
x2 = x.clone()
for i, L in enumerate(lengths):
    x2[i, L:] = torch.randint(1, VOCAB, (MAXLEN - int(L),))   # パディング部を別トークンに

model.eval()
with torch.no_grad():
    a, b = model(x, lengths), model(x2, lengths)
print("最大差:", (a - b).abs().max().item(), "← 0 なら梱包が効いている")

最大差: 0.0 ← 0 なら梱包が効いている


## 勾配クリッピングの置き場所

⚠️ **RNN では勾配クリッピングを入れるのが定石**（勾配爆発の防止）。
学習ループの中では、**逆伝播と更新のあいだ**にこの 1 行を挟む。

In [6]:
model.train()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

for step in range(3):
    optimizer.zero_grad()
    loss = criterion(model(x, lengths), y)

    loss.backward()
    total_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    print(f"step {step}: loss={loss.item():.4f}  クリップ前の勾配ノルム={total_norm:.3f}")

step 0: loss=0.6980  クリップ前の勾配ノルム=0.221
step 1: loss=0.6308  クリップ前の勾配ノルム=0.233
step 2: loss=0.5628  クリップ前の勾配ノルム=0.304


## 読み取りどころ

- **`padding_idx=0` は「0 番のトークンは埋め草」という宣言**である。Embedding の勾配が 0 番には流れなくなる。
- **`pack_padded_sequence` は速度だけの話ではない。**梱包しないと LSTM が埋め草まで読み込み、**最終隠れ状態が埋め草に汚染される**。上の「最大差」が 0 になることが、それを防げている証拠。
- **`h[-2]` と `h[-1]` は最終層の順方向・逆方向。**`h` の形は (層数×方向, バッチ, 隠れ次元) なので、最後の 2 つが最終層の両方向になる。
- **`clip_grad_norm_` は `backward()` の後・`step()` の前。**前に置くと勾配がまだ無く、後に置くと更新済みで手遅れになる。

⚠️ 返り値の `total_norm` は**クリップ前**のノルム。これを毎ステップ記録しておくと、勾配爆発が起きているかを後から追える。